In [2]:
%pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

  Using cached mysql_connector_python-26.10.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
Using cached mysql_connector_python-26.10.0-cp313-cp313-win_amd64.whl (17.7 MB)


In [3]:
import numpy as np
import pandas as pd
import mysql.connector
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [4]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

In [5]:
conn.is_connected()

True

In [6]:
# create a function out of this code to connect to the database and return the connection object
def connect_to_db():
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="sharif123",
        database="supply_chain"
    )
    return conn

In [7]:
connect_to_db().is_connected()

True

In [8]:
db = connect_to_db()
cursor = db.cursor(dictionary=True)

In [9]:
cursor

In [10]:
cursor.execute("select count(*) as total_suppliers from suppliers")

In [11]:
row = cursor.fetchone()

In [12]:
list(row.values())[0]

50

In [11]:
queries = {
"Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

"Total Products": "SELECT COUNT(*) AS count FROM products",

"Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

"Total Sale Value (Last 3 Months)": """
SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) AS total_sale
FROM stock_entries se
JOIN products p ON se.product_id = p.product_id
WHERE se.change_type = 'Sale'
AND se.entry_date >= (
SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
""",

"Total Restock Value (Last 3 Months)": """
SELECT ROUND(SUM(se.change_quantity * p.price), 2) AS total_restock
FROM stock_entries se
JOIN products p ON se.product_id = p.product_id
WHERE se.change_type = 'Restock'
AND se.entry_date >= (
SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
""",

"Below Reorder & No Pending Reorders": """
SELECT COUNT(*) AS below_reorder
FROM products p
WHERE p.stock_quantity < p.reorder_level
AND p.product_id NOT IN (
SELECT DISTINCT product_id FROM reorders WHERE status = 'Pending')
"""
}

In [12]:
result = {}
for label, query in queries.items():
    cursor.execute(query)
    row = cursor.fetchone()
    result[label] = list(row.values())[0]

In [13]:
for label, value in result.items():
    print(f"{label}: {value}")

Total Suppliers: 50
Total Products: 200
Total Categories Dealing: 5
Total Sale Value (Last 3 Months): 1529482.99
Total Restock Value (Last 3 Months): 6691663.1
Below Reorder & No Pending Reorders: 14


In [14]:
## Now lets create function out of it 

def get_basic_info(cursor):
    """
    Retrieve summary inventory and supply chain metrics.

    Args:
        cursor (mysql.connector.cursor.MySQLCursorDict): Cursor object with dictionary=True.

    Returns:
        dict: Dictionary of metric labels and their values.
    """

    queries = {
        "Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

        "Total Products": "SELECT COUNT(*) AS count FROM products",

        "Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

        "Total Sale Value (Last 3 Months)": """
            SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) AS total_sale
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Sale'
              AND se.entry_date >= (
                  SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Total Restock Value (Last 3 Months)": """
            SELECT ROUND(SUM(se.change_quantity * p.price), 2) AS total_restock
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Restock'
              AND se.entry_date >= (
                  SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Below Reorder & No Pending Reorders": """
            SELECT COUNT(*) AS below_reorder
            FROM products p
            WHERE p.stock_quantity < p.reorder_level
              AND p.product_id NOT IN (
                  SELECT DISTINCT product_id FROM reorders WHERE status = 'Pending')
        """
    }

    results = {}
    for label, query in queries.items():
        cursor.execute(query)
        row = cursor.fetchone()
        # Since row is a dictionary, extract the single value by getting the first value in dict.values()
        results[label] = list(row.values())[0]

    return results

In [15]:
queries = {
        "Suppliers Contact Details": "SELECT supplier_name, contact_name, email, phone FROM suppliers",

        "Products with Supplier and Stock": """
            SELECT 
                p.product_name,
                s.supplier_name,
                p.stock_quantity,
                p.reorder_level
            FROM products p
            JOIN suppliers s ON p.supplier_id = s.supplier_id
            ORDER BY p.product_name ASC
        """,

        "Products Needing Reorder": """
            SELECT product_name, stock_quantity, reorder_level
            FROM products
            WHERE stock_quantity <= reorder_level
        """
    }

tables = {}
for label, query in queries.items():
    cursor.execute(query)
    tables[label] = cursor.fetchall()

In [16]:
tables

{'Suppliers Contact Details': [{'supplier_name': 'Anderson-Thompson',
   'contact_name': 'Bonnie Davis',
   'email': 'zacharysanchez@hotmail.com',
   'phone': '829.485.9853x0522'},
  {'supplier_name': 'Rowland Ltd',
   'contact_name': 'Beth Stevens',
   'email': 'joshua60@yahoo.com',
   'phone': '(779)942-0726'},
  {'supplier_name': 'Baxter-Meadows',
   'contact_name': 'Lisa Lewis',
   'email': 'andersonchristina@yahoo.com',
   'phone': '449-766-7325'},
  {'supplier_name': 'Wilson, Graham and Williams',
   'contact_name': 'David Martinez',
   'email': 'abrown@hotmail.com',
   'phone': '+1-653-827-5215x266'},
  {'supplier_name': 'Smith, Kennedy and Moreno',
   'contact_name': 'Victoria Gonzalez',
   'email': 'mary30@williams-moore.com',
   'phone': '891-859-2775x35297'},
  {'supplier_name': 'Middleton LLC',
   'contact_name': 'Megan Miller',
   'email': 'kelseywilliams@gmail.com',
   'phone': '001-724-731-6199x4596'},
  {'supplier_name': 'Evans Inc',
   'contact_name': 'Danielle Moore',

In [17]:
def get_additonal_tables(cursor):
    queries = {
        "Suppliers Contact Details": "SELECT supplier_name, contact_name, email, phone FROM suppliers",

        "Products with Supplier and Stock": """
            SELECT 
                p.product_name,
                s.supplier_name,
                p.stock_quantity,
                p.reorder_level
            FROM products p
            JOIN suppliers s ON p.supplier_id = s.supplier_id
            ORDER BY p.product_name ASC
        """,

        "Products Needing Reorder": """
            SELECT product_name, stock_quantity, reorder_level
            FROM products
            WHERE stock_quantity <= reorder_level
        """
    }

    tables = {}
    for label, query in queries.items():
        cursor.execute(query)
        tables[label] = cursor.fetchall()

    return tables

In [18]:
get_additonal_tables(cursor)

{'Suppliers Contact Details': [{'supplier_name': 'Anderson-Thompson',
   'contact_name': 'Bonnie Davis',
   'email': 'zacharysanchez@hotmail.com',
   'phone': '829.485.9853x0522'},
  {'supplier_name': 'Rowland Ltd',
   'contact_name': 'Beth Stevens',
   'email': 'joshua60@yahoo.com',
   'phone': '(779)942-0726'},
  {'supplier_name': 'Baxter-Meadows',
   'contact_name': 'Lisa Lewis',
   'email': 'andersonchristina@yahoo.com',
   'phone': '449-766-7325'},
  {'supplier_name': 'Wilson, Graham and Williams',
   'contact_name': 'David Martinez',
   'email': 'abrown@hotmail.com',
   'phone': '+1-653-827-5215x266'},
  {'supplier_name': 'Smith, Kennedy and Moreno',
   'contact_name': 'Victoria Gonzalez',
   'email': 'mary30@williams-moore.com',
   'phone': '891-859-2775x35297'},
  {'supplier_name': 'Middleton LLC',
   'contact_name': 'Megan Miller',
   'email': 'kelseywilliams@gmail.com',
   'phone': '001-724-731-6199x4596'},
  {'supplier_name': 'Evans Inc',
   'contact_name': 'Danielle Moore',

In [ ]:
def add_new_manual_id(cursor, db, p_name, p_category, p_price, p_stock_quantity, p_reorder_level, p_supplier_id):
    """
    Add a new product to the products table with a manually specified product_id.

    Args:
        cursor (mysql.connector.cursor.MySQLCursorDict): Cursor object with dictionary=True.
        db (mysql.connector.connection.MySQLConnection): Database connection object.
        p_name (str): Name of the product.
        p_category (str): Category of the product.
        p_price (float): Price of the product.
        p_stock_quantity (int): Stock quantity of the product.
        p_reorder_level (int): Reorder level of the product.
        p_supplier_id (int): Supplier ID associated with the product.

    Returns:
        None
    """
    proc_call = "call AddNewProductManualID(%s, %s, %s, %s, %s, %s)"
    params = (p_name, p_category, p_price, p_stock_quantity, p_reorder_level, p_supplier_id)
    cursor.execute(proc_call, params)
    db.commit()

In [13]:
def get_categories(cursor):
    """
    Retrieve distinct product categories from the products table.

    Args:
        cursor (mysql.connector.cursor.MySQLCursorDict): Cursor object with dictionary=True.

    Returns:
        list: List of distinct product categories.
    """
    query = "SELECT DISTINCT category FROM products ORDER BY category ASC"
    cursor.execute(query)
    rows = cursor.fetchall()
    categories = [row['category'] for row in rows]
    return categories

In [14]:
get_categories(cursor)

['Clothing', 'Electronics', 'Furniture', 'Groceries', 'Toys']

In [15]:
def get_suppliers(cursor):
    """
    Retrieve distinct supplier names from the suppliers table.

    Args:
        cursor (mysql.connector.cursor.MySQLCursorDict): Cursor object with dictionary=True.

    Returns:
        list: List of distinct supplier names.
    """
    query = "SELECT supplier_id, supplier_name FROM suppliers ORDER BY supplier_name ASC"
    cursor.execute(query)
    rows = cursor.fetchall()
    return rows

In [16]:
get_suppliers(cursor)

[{'supplier_id': 1, 'supplier_name': 'Anderson-Thompson'},
 {'supplier_id': 48, 'supplier_name': 'Armstrong-Vance'},
 {'supplier_id': 24, 'supplier_name': 'Barker, White and Carson'},
 {'supplier_id': 25, 'supplier_name': 'Barrett Ltd'},
 {'supplier_id': 3, 'supplier_name': 'Baxter-Meadows'},
 {'supplier_id': 19, 'supplier_name': 'Charles Inc'},
 {'supplier_id': 41, 'supplier_name': 'Clark Group'},
 {'supplier_id': 23, 'supplier_name': 'Douglas Ltd'},
 {'supplier_id': 32, 'supplier_name': 'Elliott-Ayers'},
 {'supplier_id': 7, 'supplier_name': 'Evans Inc'},
 {'supplier_id': 27, 'supplier_name': 'Franklin, Kane and Price'},
 {'supplier_id': 15, 'supplier_name': 'Freeman-Gordon'},
 {'supplier_id': 13, 'supplier_name': 'Gallagher-Miller'},
 {'supplier_id': 17, 'supplier_name': 'Gomez PLC'},
 {'supplier_id': 40, 'supplier_name': 'Hall-Brown'},
 {'supplier_id': 38, 'supplier_name': 'Harris-Cummings'},
 {'supplier_id': 42, 'supplier_name': 'Henderson LLC'},
 {'supplier_id': 50, 'supplier_name